In [ ]:
# =============================================================================
# ReyStego v2 — Reversible Coverless Multi Lossless Steganography
#              with Encryption, Compression, and IWT Subband Cover Mining
#
# Cover generation : SDXL-Base 1.0 (single pipeline, no refiner)
# Mining strategy  : Pre-alignment rate rho maximisation over M candidates
# Embedding        : IWT Subband LSB — ALL 3 detail subbands (LH+HL+HH)
# Encryption       : ChaCha20-Poly1305 (AEAD) + scrypt KDF
# Compression      : Integer Haar Wavelet Transform (1-D) + LZMA preset 9
# Secret sharing   : zfec erasure coding (K-of-N)
# Permutation seed : Derived from password via scrypt (never public)
#
# ── Dual-IWT Architecture ────────────────────────────────────────────────────
#
#   Layer 1 — SecureProcessor (1-D IWT, secret side):
#     The secret image bytes are losslessly compressed using the Integer Haar
#     Wavelet Transform (1-D lifting), followed by LZMA-9 compression, then
#     ChaCha20-Poly1305 authenticated encryption.  This pipeline is lossless
#     and perfectly invertible.  Output is an encrypted byte payload.
#
#   Layer 2 — IWTSubbandEmbedder (2-D IWT, cover side):
#     The encrypted payload bits are embedded into the detail subbands of the
#     AI-generated cover image via IWT-2D LSB replacement.  This is entirely
#     independent of Layer 1.  The cover's lossless PNG storage ensures
#     bit-exact IWT round-trips, preserving the embedded payload.
#
#   These two IWT layers are complementary and non-conflicting:
#     IWT-1D compresses the secret; IWT-2D hides the compressed secret.
# =============================================================================

import os
import io
import random
import getpass
import warnings
import struct
import dataclasses
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

warnings.filterwarnings('ignore')

WORK_DIR        = os.environ.get('KAGGLE_WORKDIR', '/kaggle/working')
os.makedirs(WORK_DIR, exist_ok=True)
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

COVER_RES       = 1024   # native SDXL resolution
MAX_RETRIES     = 5      # outer retries per share (new cover each time)
MINE_CANDIDATES = 10     # M: candidates per share during cover mining
INFERENCE_STEPS = 20     # DPM-Solver++ steps (sufficient for SDXL-Base)

NEGATIVE_PROMPT = (
    'blurry, low quality, watermarks, text, deformed, ugly, '
    'extra limbs, bad anatomy, duplicate'
)

# --- Dependencies ------------------------------------------------------------
try:
    import zfec
    import lzma
    from Crypto.Cipher       import ChaCha20_Poly1305
    from Crypto.Random       import get_random_bytes
    from Crypto.Protocol.KDF import scrypt
    from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
except ImportError:
    import subprocess
    print('Installing dependencies...')
    subprocess.run(
        ['pip', 'install', 'zfec', 'pycryptodome',
         'diffusers', 'transformers', 'accelerate'],
        check=False
    )
    import zfec, lzma
    from Crypto.Cipher       import ChaCha20_Poly1305
    from Crypto.Random       import get_random_bytes
    from Crypto.Protocol.KDF import scrypt
    from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

print(f'Device: {DEVICE}')


# =============================================================================
# EMBED STATISTICS
# =============================================================================
@dataclasses.dataclass
class EmbedStats:
    """Embedding quality metrics returned by IWTSubbandEmbedder.embed()."""
    n_bits              : int
    rho_pre             : float   # pre-alignment rate (bits not needing change)
    n_modified          : int     # coefficients changed in primary embedding
    n_clipping_corrected: int     # bits corrected by in-embedder post-correction
    accuracy            : float   # final verified bit accuracy (1.0 = perfect)

    def __str__(self) -> str:
        return (
            f'Bits: {self.n_bits:,} | '
            f'Pre-aligned: {self.rho_pre:.2%} | '
            f'Modified: {self.n_modified:,} ({self.n_modified/max(self.n_bits,1):.2%}) | '
            f'Clipping corrections: {self.n_clipping_corrected} | '
            f'Accuracy: {self.accuracy:.6f}'
        )


# =============================================================================
# GENERATIVE MODEL — SDXL-Base 1.0 (single pipeline, refiner removed)
# =============================================================================
pipe = None
try:
    print('Loading SDXL-Base 1.0 (no refiner)...')
    pipe = StableDiffusionXLPipeline.from_pretrained(
        'stabilityai/stable-diffusion-xl-base-1.0',
        torch_dtype=torch.float16,
        variant='fp16',
        use_safetensors=True
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        pipe.scheduler.config, use_karras_sigmas=True
    )
    pipe.set_progress_bar_config(disable=True)
    if DEVICE == 'cuda':
        pipe.enable_model_cpu_offload()
        pipe.enable_vae_slicing()
    print('Model ready.')
except Exception as e:
    print(f'Model load error: {e}')
    print('Falling back to random-noise covers.')


# =============================================================================
# INTEGER HAAR WAVELET TRANSFORM (1-D)  — Layer 1: Secret compression
# =============================================================================
class IntegerHaarTransform:
    """1-D lossless integer Haar lifting step — perfectly invertible."""

    def forward(self, data_bytes: bytes) -> bytes:
        arr    = np.frombuffer(data_bytes, dtype=np.uint8).astype(np.int16)
        if len(arr) % 2:
            arr = np.append(arr, 0)
        detail = arr[1::2] - arr[0::2]
        avg    = arr[0::2] + np.floor_divide(detail, 2)
        return np.concatenate((avg, detail)).astype(np.int16).tobytes()

    def inverse(self, data_bytes: bytes) -> bytes:
        arr    = np.frombuffer(data_bytes, dtype=np.int16)
        half   = len(arr) // 2
        avg, detail = arr[:half], arr[half:]
        even   = avg - np.floor_divide(detail, 2)
        odd    = detail + even
        rec    = np.zeros(len(even) + len(odd), dtype=np.int16)
        rec[0::2] = even
        rec[1::2] = odd
        return rec.astype(np.uint8).tobytes()


# =============================================================================
# SECURE PROCESSOR — Layer 1: IWT-1D → LZMA-9 → ChaCha20-Poly1305
# =============================================================================
class SecureProcessor:
    def __init__(self, password: str):
        self.iwt          = IntegerHaarTransform()
        self.password     = password.encode('utf-8')
        self.lzma_filters = [{'id': lzma.FILTER_LZMA2, 'preset': 9}]

    def _derive_key(self, salt: bytes) -> bytes:
        return scrypt(self.password, salt, 32, N=2**14, r=8, p=1)

    def pack(self, image_path: str) -> tuple:
        """Returns (encrypted_payload_bytes, original_image_size)."""
        img       = Image.open(image_path).convert('L')
        raw       = img.tobytes()
        size      = img.size
        iwt_data  = self.iwt.forward(raw)
        comp      = lzma.compress(iwt_data, format=lzma.FORMAT_RAW,
                                   filters=self.lzma_filters)
        salt            = get_random_bytes(16)
        key             = self._derive_key(salt)
        cipher          = ChaCha20_Poly1305.new(key=key)
        ct, tag         = cipher.encrypt_and_digest(comp)
        return salt + cipher.nonce + tag + ct, size

    def unpack(self, payload: bytes, size) -> Image.Image:
        """Decrypts and reconstructs the secret image. Returns None on failure."""
        try:
            salt, nonce, tag = payload[:16], payload[16:28], payload[28:44]
            key    = self._derive_key(salt)
            cipher = ChaCha20_Poly1305.new(key=key, nonce=nonce)
            comp   = cipher.decrypt_and_verify(payload[44:], tag)
            d      = lzma.LZMADecompressor(format=lzma.FORMAT_RAW,
                                            filters=self.lzma_filters)
            raw    = self.iwt.inverse(d.decompress(comp))
            return Image.frombytes('L', size, raw[:size[0]*size[1]])
        except Exception:
            return None


# =============================================================================
# IWT SUBBAND LSB EMBEDDER  — Layer 2: Wavelet-domain cover embedding
# =============================================================================
# Embedding pool:
#   pool = [LH_R | LH_G | LH_B | HL_R | HL_G | HL_B | HH_R | HH_G | HH_B]
#
# LH/HL subbands: each coefficient change causes at most ±1 pixel change in
#   spatial domain → guaranteed to stabilise after one PNG round-trip.
# HH subband: each coefficient change causes ±1 change in up to 4 pixels
#   (2×2 block via IIWT2D) → can create cascading clipping cycles.
#
# Clip-safety strategy (permutation priority):
#   _full_perm() returns LH+HL positions FIRST, HH positions LAST.
#   Since payload << LH+HL capacity for typical medical images, HH positions
#   are never selected, eliminating cascading clip cycles entirely.
#   Capacity per 1024×1024 share:
#     LH+HL alone : 1,572,864 bits = 196,608 bytes  (guaranteed clip-safe)
#     Full pool   : 2,359,296 bits = 294,912 bytes  (used if payload is large)
# =============================================================================
class IWTSubbandEmbedder:
    """
    Lossless wavelet-domain steganographic embedder.
    Uses all three detail subbands (LH + HL + HH) across all RGB channels.
    LH+HL positions are prioritised to avoid cascading HH clipping cycles.
    """

    def __init__(self, perm_seed: int = 42):
        self.perm_seed = perm_seed

    # ------------------------------------------------------------------
    def _iwt2d(self, channel: np.ndarray) -> tuple:
        ch = channel.astype(np.int32)
        H, W = ch.shape
        if H % 2: ch = np.vstack([ch, ch[-1:]])
        if W % 2: ch = np.hstack([ch, ch[:, -1:]])
        L  = (ch[:, 0::2] + ch[:, 1::2]) >> 1
        D  =  ch[:, 1::2] - ch[:, 0::2]
        LL = (L[0::2] + L[1::2]) >> 1
        HL =  L[1::2] - L[0::2]
        LH = (D[0::2] + D[1::2]) >> 1
        HH =  D[1::2] - D[0::2]
        return LL, LH, HL, HH

    # ------------------------------------------------------------------
    def _iiwt2d(self, LL, LH, HL, HH, orig_H, orig_W) -> np.ndarray:
        L_even = LL - (HL >> 1)
        L_odd  = L_even + HL
        L = np.empty((LL.shape[0] * 2, LL.shape[1]), dtype=np.int32)
        L[0::2] = L_even;  L[1::2] = L_odd
        D_even = LH - (HH >> 1)
        D_odd  = D_even + HH
        D = np.empty((LH.shape[0] * 2, LH.shape[1]), dtype=np.int32)
        D[0::2] = D_even;  D[1::2] = D_odd
        ch = np.empty((L.shape[0], L.shape[1] * 2), dtype=np.int32)
        ch[:, 0::2] = L - (D >> 1)
        ch[:, 1::2] = ch[:, 0::2] + D
        return np.clip(ch[:orig_H, :orig_W], 0, 255)

    # ------------------------------------------------------------------
    def _pool(self, image: Image.Image) -> tuple:
        arr  = np.array(image.convert('RGB'))
        H, W = arr.shape[:2]
        meta = []
        lh_parts, hl_parts, hh_parts = [], [], []
        for c in range(3):
            LL, LH, HL, HH = self._iwt2d(arr[:, :, c])
            lh_parts.append(LH.ravel().copy())
            hl_parts.append(HL.ravel().copy())
            hh_parts.append(HH.ravel().copy())
            meta.append((LL, LH.shape, HL.shape, HH.shape, H, W))
        lh_cat = np.concatenate(lh_parts)
        hl_cat = np.concatenate(hl_parts)
        hh_cat = np.concatenate(hh_parts)
        pool   = np.concatenate([lh_cat, hl_cat, hh_cat]).astype(np.int32)
        return pool, meta, len(lh_cat), len(hl_cat), len(hh_cat)

    # ------------------------------------------------------------------
    def _reconstruct(self, pool, meta, lh_len, hl_len, hh_len) -> np.ndarray:
        ch_lh  = lh_len // 3
        ch_hl  = hl_len // 3
        ch_hh  = hh_len // 3
        OH, OW = meta[0][4], meta[0][5]
        result = np.zeros((OH, OW, 3), dtype=np.uint8)
        for c in range(3):
            LL, lh_shape, hl_shape, hh_shape, oh, ow = meta[c]
            LH = pool[c*ch_lh:(c+1)*ch_lh].reshape(lh_shape)
            HL = pool[lh_len+c*ch_hl:lh_len+(c+1)*ch_hl].reshape(hl_shape)
            HH = pool[lh_len+hl_len+c*ch_hh:lh_len+hl_len+(c+1)*ch_hh].reshape(hh_shape)
            result[:, :, c] = self._iiwt2d(LL, LH, HL, HH, oh, ow)
        return result

    # ------------------------------------------------------------------
    def _full_perm(self, pool_size: int, lh_hl_size: int = None) -> np.ndarray:
        """
        Deterministic permutation with LH+HL positions FIRST, HH positions LAST.

        LH+HL subbands cause at most ±1 pixel change per coefficient — they
        always stabilise after one PNG round-trip with no clipping cycles.
        HH causes ±1 coefficient change but affects a 2×2 pixel block (up to
        ±2 total pixel change), which can cascade.  By placing HH positions at
        the end of the permutation, they are only used when the payload exceeds
        LH+HL capacity — i.e., essentially never for typical medical images.
        """
        if lh_hl_size is None:
            lh_hl_size = (pool_size * 2) // 3   # LH+HL = 2/3 of pool
        hh_size = pool_size - lh_hl_size
        rng = np.random.default_rng(self.perm_seed)
        lh_hl_perm = rng.permutation(lh_hl_size)
        hh_perm    = rng.permutation(hh_size) + lh_hl_size
        return np.concatenate([lh_hl_perm, hh_perm])

    # ------------------------------------------------------------------
    def capacity(self, image: Image.Image) -> int:
        arr  = np.array(image.convert('RGB'))
        H, W = arr.shape[:2]
        h2   = (H + H % 2) // 2
        w2   = (W + W % 2) // 2
        return 3 * h2 * w2 * 3

    # ------------------------------------------------------------------
    def alignment_score(self, image: Image.Image,
                         target_bits: np.ndarray) -> float:
        extracted = self.extract(image, len(target_bits))
        return float(np.mean(extracted == target_bits.astype(np.uint8)))

    # ------------------------------------------------------------------
    def embed(self, cover: Image.Image, payload_bits: np.ndarray) -> tuple:
        """
        Embed payload_bits into cover via wavelet-domain LSB replacement.

        Clipping-safe post-correction uses a convergence-detecting loop:
        rounds continue until n_bad reaches 0 OR stops improving (cycle
        detected), at which point further rounds cannot help.  For payloads
        within LH+HL capacity this loop almost never triggers.

        Returns: (stego_image, EmbedStats)
        """
        pool, meta, lh_len, hl_len, hh_len = self._pool(cover)
        n_bits    = len(payload_bits)
        pool_size = len(pool)

        if n_bits > pool_size:
            raise ValueError(
                f'Payload too large: {n_bits} bits > pool capacity {pool_size}'
            )

        full_perm = self._full_perm(pool_size, lh_len + hl_len)
        positions = full_perm[:n_bits].copy()
        pb = payload_bits.astype(np.uint8)

        # Pre-alignment stats
        pre_lsbs   = (pool[positions] & 1).astype(np.uint8)
        rho_pre    = float(np.mean(pre_lsbs == pb))
        n_modified = int(np.sum(pre_lsbs != pb))

        # Primary LSB embedding
        pool[positions] = (pool[positions] & ~1) | pb.astype(np.int32)
        stego = Image.fromarray(
            self._reconstruct(pool, meta, lh_len, hl_len, hh_len)
        )

        # Convergence-detecting clipping correction loop.
        # Exits immediately on n_bad == 0 (perfect) or when n_bad stops
        # decreasing (cycle — more rounds won't help, let outer retry handle it).
        n_clipping_corrected = 0
        prev_bad = n_bits + 1

        for _round in range(10):
            buf = io.BytesIO()
            stego.save(buf, format='PNG')
            buf.seek(0)
            pool_rt, meta_rt, lh_rt, hl_rt, hh_rt = self._pool(Image.open(buf))
            extracted_rt = (pool_rt[positions] & 1).astype(np.uint8)
            bad_mask     = (extracted_rt != pb)
            n_bad        = int(np.sum(bad_mask))

            if n_bad == 0:
                break
            if n_bad >= prev_bad:
                break   # no improvement — stuck in cycle, stop here
            prev_bad = n_bad
            n_clipping_corrected += n_bad

            pool_rt[positions[bad_mask]] = (
                (pool_rt[positions[bad_mask]] & ~1)
                | pb[bad_mask].astype(np.int32)
            )
            stego = Image.fromarray(
                self._reconstruct(pool_rt, meta_rt, lh_rt, hl_rt, hh_rt)
            )

        # Final accuracy verification
        buf = io.BytesIO()
        stego.save(buf, format='PNG')
        buf.seek(0)
        final_ext = self.extract(Image.open(buf), n_bits)
        accuracy  = float(np.mean(final_ext == pb))

        stats = EmbedStats(
            n_bits=n_bits,
            rho_pre=rho_pre,
            n_modified=n_modified,
            n_clipping_corrected=n_clipping_corrected,
            accuracy=accuracy,
        )
        return stego, stats

    # ------------------------------------------------------------------
    def extract(self, stego: Image.Image, n_bits: int) -> np.ndarray:
        """Coverless extraction — no original cover needed."""
        pool, _, lh_len, hl_len, _ = self._pool(stego)
        positions = self._full_perm(len(pool), lh_len + hl_len)[:n_bits]
        return (pool[positions] & 1).astype(np.uint8)


# =============================================================================
# COVER MINER
# =============================================================================
class CoverMiner:
    """Mines the best AI-generated cover for a given payload bitstream."""

    def __init__(self, engine, embedder: IWTSubbandEmbedder,
                  num_candidates: int = MINE_CANDIDATES):
        self.engine         = engine
        self.embedder       = embedder
        self.num_candidates = max(1, num_candidates)   # guard against 0

    def mine(self, target_bits: np.ndarray, prompt: str) -> tuple:
        """Return (best_image, best_seed, best_rho)."""
        best_rho, best_image, best_seed = -1.0, None, None
        print(f'    Mining {self.num_candidates} candidates...')
        for i in range(self.num_candidates):
            seed      = random.randint(0, 2**32 - 1)
            candidate = self.engine.generate_cover(prompt, seed)
            rho       = self.embedder.alignment_score(candidate, target_bits)
            flag      = ' <- best' if rho > best_rho else ''
            print(f'      [{i+1:2d}/{self.num_candidates}]  rho={rho:.4f}{flag}')
            if rho > best_rho:
                best_rho, best_image, best_seed = rho, candidate, seed
        print(f'    Selected: rho={best_rho:.4f}  seed={best_seed}')
        return best_image, best_seed, best_rho


# =============================================================================
# STEGO ENGINE
# =============================================================================
class StegoEngine:
    """
    Orchestrates the full ReyStego pipeline.
    embed()   : Erasure-encode → Mine → IWT-embed → verify
    recover() : IWT-extract → Erasure-decode → Decrypt
    """
    HEADER_BITS = 32   # 4-bit share_id | 28-bit payload_length_bytes

    def __init__(self, password: str):
        self.password = password
        perm_bytes = scrypt(
            password.encode('utf-8'),
            b'reystego-perm-v2',
            4, N=2**14, r=8, p=1
        )
        perm_seed     = struct.unpack('>I', perm_bytes)[0]
        self.embedder = IWTSubbandEmbedder(perm_seed=perm_seed)

    # ------------------------------------------------------------------
    def generate_cover(self, prompt: str, seed: int) -> Image.Image:
        """
        Single SDXL-Base generation with CUDA OOM recovery.
        Falls back to random noise if all SDXL attempts fail.
        """
        if pipe is None:
            rng = np.random.default_rng(seed)
            return Image.fromarray(
                rng.integers(0, 255, (COVER_RES, COVER_RES, 3), dtype=np.uint8)
            )
        for _try in range(3):
            try:
                gen = torch.Generator(device=DEVICE).manual_seed(seed)
                out = pipe(
                    prompt=prompt + ', masterpiece, 8k, highly detailed, rich texture',
                    negative_prompt=NEGATIVE_PROMPT,
                    num_inference_steps=INFERENCE_STEPS,
                    height=COVER_RES, width=COVER_RES,
                    generator=gen
                )
                if out.images:
                    return out.images[0]
            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    torch.cuda.empty_cache()
                    seed = (seed + 1) & 0xFFFFFFFF
                    continue
                raise
        # All SDXL attempts failed — fall back to deterministic noise
        rng = np.random.default_rng(seed)
        return Image.fromarray(
            rng.integers(0, 255, (COVER_RES, COVER_RES, 3), dtype=np.uint8)
        )

    # ------------------------------------------------------------------
    def _make_bitstream(self, share_id: int, payload_bytes: bytes) -> np.ndarray:
        hval = (share_id & 0xF) << 28 | (len(payload_bytes) & 0xFFFFFFF)
        hdr  = np.unpackbits(
            np.frombuffer(struct.pack('>I', hval), dtype=np.uint8), bitorder='big'
        )
        body = np.unpackbits(
            np.frombuffer(payload_bytes, dtype=np.uint8), bitorder='big'
        )
        return np.concatenate((hdr, body))

    # ------------------------------------------------------------------
    def embed(self, payload: bytes, prefix: str, prompt: str,
               zfec_k: int, zfec_n: int) -> tuple:
        """
        Full embedding pipeline. Returns (file_paths, stego_images).
        Each share retries up to MAX_RETRIES times with a fresh cover.
        With the prioritised permutation (LH+HL first), accuracy=1.0 is
        achieved on the first attempt for typical payload sizes.
        """
        cap = self.embedder.capacity(Image.new('RGB', (COVER_RES, COVER_RES)))
        lh_hl_cap = cap * 2 // 3
        print(f'Payload: {len(payload)} bytes | K={zfec_k}, N={zfec_n} | '
              f'Cover capacity: {cap // 8:,} bytes/share '
              f'(LH+HL safe zone: {lh_hl_cap // 8:,} bytes)')

        enc     = zfec.Encoder(zfec_k, zfec_n)
        pad_len = (zfec_k - len(payload) % zfec_k) % zfec_k
        padded  = payload + b'\x00' * pad_len
        bsz     = len(padded) // zfec_k
        shares  = enc.encode([padded[i:i+bsz] for i in range(0, len(padded), bsz)])

        miner       = CoverMiner(self, self.embedder)
        files, imgs = [], []

        for i, share_data in enumerate(shares):
            bits = self._make_bitstream(i, share_data)
            print(f'\n--- Share {i+1}/{zfec_n}  ({len(bits):,} bits) ---')

            best_cover, best_seed, best_rho = miner.mine(bits, prompt)

            success = None
            for attempt in range(MAX_RETRIES):
                cover = best_cover if attempt == 0 \
                        else miner.mine(bits, prompt)[0]

                print(f'  [Attempt {attempt+1}] embedding...', end=' ')
                stego, stats = self.embedder.embed(cover, bits)
                print(stats)

                if stats.accuracy == 1.0:
                    success = stego
                    break
                errs = int(round((1 - stats.accuracy) * stats.n_bits))
                print(f'  WARNING: {errs} residual errors — retrying with new cover')

            if success:
                path = os.path.join(WORK_DIR, f'{prefix}_S{i}.png')
                success.save(path, format='PNG')
                files.append(path)
                imgs.append(success)
            else:
                print(f'  Share {i+1} failed after {MAX_RETRIES} attempts.')

        return files, imgs

    # ------------------------------------------------------------------
    def recover(self, img_path: str) -> tuple:
        """
        Blind extraction: reads the 32-bit header, then extracts the full
        bitstream.  Validates share_id before returning to prevent zfec
        decode failures from garbled headers.

        Returns: (share_id, share_data_bytes) or None on failure.
        """
        try:
            stego = Image.open(img_path)
            hbits = self.embedder.extract(stego, self.HEADER_BITS)
            hval  = struct.unpack(
                '>I', np.packbits(hbits, bitorder='big').tobytes()
            )[0]
            sid   = (hval >> 28) & 0xF
            plen  = hval & 0xFFFFFFF

            if plen <= 0 or plen > 100_000_000:
                return None

            all_bits = self.embedder.extract(stego, self.HEADER_BITS + plen * 8)
            return sid, np.packbits(all_bits[self.HEADER_BITS:], bitorder='big').tobytes()
        except Exception as e:
            print(f'    recover({os.path.basename(img_path)}): {e}')
            return None


# =============================================================================
# MAIN
# =============================================================================
if __name__ == '__main__':
    import math
    print('=== ReyStego v2 | IWT Subband Embedding (LH+HL+HH) + Cover Mining ===')

    target_file = 'Abdominal.tiff'
    input_path  = next(
        (os.path.join(d, target_file)
         for d in [WORK_DIR, '/kaggle/input/dataset', '.']
         if os.path.exists(os.path.join(d, target_file))),
        None
    )

    if not input_path:
        print(f'Error: {target_file} not found.')
    else:
        print(f'Input : {input_path}')
        user_prompt = input('Prompt for cover images: ').strip() or 'abstract texture'

        try:
            zfec_n = int(input('Total shares N [default 5]: ').strip() or '5')
            zfec_k = int(input('Required shares K [default 3]: ').strip() or '3')
            if zfec_k > zfec_n:
                print('K > N — resetting to defaults.'); zfec_n, zfec_k = 5, 3
            if zfec_n > 16:
                print('N capped at 16.'); zfec_n, zfec_k = 16, min(zfec_k, 16)
        except ValueError:
            zfec_n, zfec_k = 5, 3

        print(f'Config: N={zfec_n}, K={zfec_k}')
        user_pass = getpass.getpass('Password: ')

        proc   = SecureProcessor(user_pass)
        engine = StegoEngine(user_pass)

        print('\n[1] Packing secret image (IWT-1D + LZMA + ChaCha20-Poly1305)...')
        payload, orig_size = proc.pack(input_path)
        print(f'    Packed: {len(payload):,} bytes')

        print('\n[2] Embedding with IWT-2D subband LSB + mining...')
        files, imgs = engine.embed(payload, 'Sec', user_prompt, zfec_k, zfec_n)

        if imgs:
            fig, axes = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
            if len(imgs) == 1: axes = [axes]
            for ax, img, f in zip(axes, imgs, files):
                ax.imshow(img); ax.axis('off')
                ax.set_title(os.path.basename(f), fontsize=9)
            plt.suptitle('Stego covers (IWT-2D LH+HL+HH LSB, mined)', fontsize=11)
            plt.tight_layout(); plt.show()

        # Guard: need at least K successfully embedded shares to recover
        if len(files) < zfec_k:
            print(f'\n  ERROR: Only {len(files)}/{zfec_n} shares embedded successfully '
                  f'(need {zfec_k}). Increase MAX_RETRIES or re-run.')
        else:
            print('\n[3] Recovering from random K-subset of shares...')
            idxs        = random.sample(range(len(files)), zfec_k)
            shares, ids = [], []
            for idx in idxs:
                res = engine.recover(files[idx])
                if res:
                    sid, data = res
                    # Validate share ID is in the expected range before decode
                    if 0 <= sid < zfec_n:
                        print(f'    Share {sid} | {len(data):,} bytes')
                        shares.append(data); ids.append(sid)
                    else:
                        print(f'    Skipping {os.path.basename(files[idx])}: '
                              f'invalid share ID {sid} (expected 0–{zfec_n-1})')

            if len(shares) >= zfec_k:
                try:
                    dec     = zfec.Decoder(zfec_k, zfec_n)
                    decoded = b''.join(dec.decode(shares, ids))
                    rec_img = proc.unpack(decoded[:len(payload)], orig_size)

                    if rec_img:
                        out_path = os.path.join(WORK_DIR, 'Recovered.tiff')
                        rec_img.save(out_path)
                        orig_arr = np.array(Image.open(input_path).convert('L'), float)
                        rec_arr  = np.array(rec_img, float)
                        mse      = np.mean((orig_arr - rec_arr) ** 2)
                        psnr     = (
                            20 * math.log10(255 / math.sqrt(mse))
                            if mse > 0 else float('inf')
                        )
                        print(f'\n    MSE  : {mse:.6f}')
                        print(f'    PSNR : {psnr:.2f} dB  (inf = pixel-perfect lossless)')
                        print(f'    Saved: {out_path}')
                        plt.figure(figsize=(4, 4))
                        plt.imshow(rec_img, cmap='gray')
                        plt.title('Recovered'); plt.axis('off'); plt.show()
                    else:
                        print('    Decryption failed (wrong password or corrupted share).')
                except Exception as e:
                    print(f'    Recovery error: {e}')
            else:
                print(f'    Could not recover enough valid shares '
                      f'({len(shares)}/{zfec_k} required).')


In [ ]:
# =============================================================================
# EVALUATION — PSNR + stego distortion analysis
# =============================================================================
import os, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def find_path(name, roots=['/kaggle/working', '/kaggle/input', '.']):
    for root in roots:
        for r, _, f in os.walk(root):
            if name in f: return os.path.join(r, name)
    return None

p1 = find_path('Abdominal.tiff')
p2 = find_path('Recovered.tiff', roots=['/kaggle/working'])

if p1 and p2:
    i1 = Image.open(p1).convert('L')
    i2 = Image.open(p2).convert('L')
    if i1.size != i2.size: i2 = i2.resize(i1.size)

    a1, a2 = np.array(i1, float), np.array(i2, float)
    mse    = np.mean((a1 - a2)**2)
    psnr   = 20*math.log10(255/math.sqrt(mse)) if mse > 0 else float('inf')
    print(f'MSE  : {mse:.6f}')
    print(f'PSNR : {psnr:.4f} dB  (inf = lossless pixel-perfect recovery)')

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(i1, cmap='gray'); ax[0].set_title('Original');  ax[0].axis('off')
    ax[1].imshow(i2, cmap='gray'); ax[1].set_title('Recovered'); ax[1].axis('off')
    plt.tight_layout(); plt.show()

    # --- Stego distortion on first share (if available) ---
    import glob
    share_files = sorted(glob.glob('/kaggle/working/Sec_S*.png'))
    if share_files:
        s = Image.open(share_files[0])
        print(f'\nStego image: {os.path.basename(share_files[0])}')
        print(f'Size: {s.size}')
        plt.figure(figsize=(5,5))
        plt.imshow(s); plt.title('Stego image (cover + embedded bits)')
        plt.axis('off'); plt.show()
else:
    print('Files not found — run Cell 1 first.')

In [ ]:
# =============================================================================
# CLEANUP
# =============================================================================
import os, glob
for f in glob.glob('/kaggle/working/Sec_*.png'):
    os.remove(f)
print('Share files removed.')